# CircularNet Model Finetuning Guide
In this notebook, we will finetune a custom RF-DETR (Region-Focused DEtection TRansformer) model for object detection using a labeled dataset in COCO format. RF-DETR improves detection accuracy by focusing attention on spatial regions of interest, making it well-suited for complex scenes with clutter or small objects. To ensure efficient training and avoid overfitting, we incorporate key training callbacks—such as early stopping, model checkpointing, and learning rate scheduling. By the end of this notebook, you’ll have a fully trained RF-DETR model ready for evaluation and deployment.

More details [Here](https://github.com/roboflow/rf-detr/tree/1.5.2)

## Download and Install Dependancies

In [ ]:
!pip install -q rfdetr==1.5.2 supervision tensorboard
!wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/ModelRegistry_432x432_March26/checkpoint_best_total.pth

## Import Libraries

In [4]:
import os
import torch
import warnings
from rfdetr import RFDETRSegMedium


os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

warnings.filterwarnings("ignore", category=RuntimeWarning, module="albumentations")
warnings.filterwarnings("ignore", message="Grad strides do not match bucket view strides")

## Structure Training Data

RF-DETR expects the dataset to be in COCO format. Divide your dataset into three subdirectories: train, valid, and test. Each subdirectory should contain its own _annotations.coco.json file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

The annotated COCO JSON files should be in the format mentioned in the link - [click here](https://roboflow.com/formats/coco-json?ref=blog.roboflow.com)

In [ ]:
! wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/sample_training_dataset.zip
! unzip -q sample_training_dataset.zip -d sample_dataset

# Additional Note : I have resized the images and annotations to 432x432, this way we can run it on lesser GPU memory

## Define Variables

In [ ]:
PRETRAINED_WEIGHTS = "./checkpoint_best_total.pth"
DATASET_DIR = "./sample_dataset/"
OUTPUT_DIR = "./checkpoints/"

EPOCHS=10 # Total no of epochs
BATCH_SIZE=2 # This is per device bs, Total bs = batch_size x grad_accum_steps x no of gpu's
GRAD_ACCUM_STEPS=2 # Gradient accumulation per gpu device
NUM_WORKERS=9  # Number of CPU workers to use
EARLY_STOPPING_PATIENCE=3 # This informs model to stop early if no improvment while training

AUG_CONFIG = {
    "HorizontalFlip": {"p": 0.2},
    "VerticalFlip": {"p": 0.2},
    "Rotate": {"limit": (90, 90), "p": 0.5},
    "GaussianBlur": {"blur_limit": 3, "p": 0.2},
    "GaussNoise": {"std_range": (0.01, 0.05), "p": 0.3},
    "RandomBrightnessContrast": {"brightness_limit": 0.2, "contrast_limit": 0.2, "p": 0.3},
    "ColorJitter": {"brightness": 0.2, "contrast": 0.2, "saturation": 0.2, "hue": 0.1, "p": 0.2},
}

## For finetuning - Load existing model

In [ ]:
model = RFDETRSegMedium(pretrain_weights=PRETRAINED_WEIGHTS)
model.class_names

[2026-05-18 08:01:54] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-18 08:01:54] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-18 08:01:55] [INFO] rf-detr - Loading pretrain weights


[2026-05-18 08:01:55] [WARNING] rf-detr - Reinitializing detection head with 50 classes based on pretrained weights, configured for 90.


In [ ]:
if __name__ == "__main__":
    model.train(
        dataset_dir=DATASET_DIR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        num_select=100,
        lr=1e-4,
        lr_encoder=1.5e-4,
        output_dir=OUTPUT_DIR,
        early_stopping=True,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=0.001,
        early_stopping_use_ema=True,
        num_workers=NUM_WORKERS,
        aug_config=AUG_CONFIG,
        warmup_epochs=2,
        lr_scheduler='cosine',
        run_test=False,
        progress_bar=True,
        distributed=False,
        tensorboard=True,
        verbose=0,
        device=str(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    )

# Monitor Model Performance using tensorboard

In [ ]:
!tensorboard --logdir=$OUTPUT_DIR --port=6006 --bind_all

## For testing the finetuned model - Please check model inference notebook : [click here](https://github.com/tensorflow/models/blob/master/official/projects/waste_identification_ml/model_inference/cn_model_run.ipynb)

In [ ]:
finetuned_model = RFDETRSegMedium(pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth")

# END of Notebook